# AIMO Progress Prize 3: Comprehensive Analysis of 50+ Experiments

**Author:** Pawan Mali  
**Competition:** [AI Mathematical Olympiad - Progress Prize 3](https://www.kaggle.com/competitions/ai-mathematical-olympiad-progress-prize-3)  
**Model:** GPT-OSS-120B (danielhanchen/gpt-oss-120b)  
**Hardware:** NVIDIA H100 (80GB)  
**Best Score:** 42/50 (V40, February 6, 2026)  

---

## Executive Summary

This writeup documents **50+ experiments** conducted over 2 months attempting to solve IMO-level mathematical problems. Our journey from 0/50 to 42/50 reveals critical insights about what works and what doesn't when using large language models for mathematical reasoning.

### Key Findings

| Finding | Impact |
|---------|--------|
| Simple 3-line prompts outperform complex prompts | +3-5 points |
| 5-component weighted entropy beats simple mean | +5-6 points |
| Temperature 1.0 with voting works best | +2-3 points |
| 131K context causes regression (not improvement) | -6 points |
| Fine-tuned model alone doesn't help | -6 points |
| Combining multiple changes usually regresses | -5-8 points |

## Table of Contents

1. [Competition Overview](#1-competition-overview)
2. [Complete Version History](#2-complete-version-history)
3. [What Works (Proven Techniques)](#3-what-works)
4. [What Doesn't Work (Failed Experiments)](#4-what-doesnt-work)
5. [Technical Deep Dive](#5-technical-deep-dive)
6. [Gap Analysis: 42 to 47](#6-gap-analysis)
7. [Key Learnings](#7-key-learnings)
8. [Recommendations](#8-recommendations)

---

## 1. Competition Overview <a name="1-competition-overview"></a>

### Constraints

- **Hardware:** Single NVIDIA H100 (80GB VRAM)
- **Time:** ~4.8 hours for 50 problems
- **Problems:** IMO-level mathematics (hidden test set)
- **Answer Format:** Non-negative integer 0-99999 in `\boxed{}`

### Our Approach

- **Model:** GPT-OSS-120B with FP8 KV cache quantization
- **Method:** Tool-Integrated Reasoning (TIR) with Python execution
- **Voting:** 8 parallel attempts with entropy-weighted voting
- **Context:** 65536 tokens maximum

---

## 2. Complete Version History <a name="2-complete-version-history"></a>

### Score Distribution Across All Submissions

```
Score Distribution (50+ submissions):

42 |█                                    (1)  V40 - BEST EVER
41 |███                                  (3)  V125, V132, V133
40 |█                                    (1)  V111
39 |████                                 (4)  Multiple
38 |███                                  (3)  Multiple
37 |█████                                (5+) V134, others
36 |██████                               (6)  V130, others
35 |██████                               (6)  V128, V131, V135, V136
34 |███                                  (3)  V41, others
33 |██                                   (2)  V126, V127
30 |█                                    (1)  
0-3|█████                                (5+) Errors/bugs
```

### Detailed Version Log

| Date | Version | Score | Key Change | Result |
|------|---------|-------|------------|--------|
| Feb 5 | V33 | ERROR | Initial setup | Dependency issues |
| Feb 5 | V39 | 39/50 | Basic TIR | First working version |
| **Feb 6** | **V40** | **42/50** | Simple prompts + weighted entropy | **BEST EVER** |
| Feb 7 | V41 | 34/50 | temp=0.5 + structured prompts | REGRESSION (-8) |
| Feb 9 | V42-V44 | 39/50 | Various tweaks | No improvement |
| Feb 10 | V45 | 39/50 | Structured prompts | No improvement |
| Feb 11-13 | V46-V48 | ERROR | Kaggle path changes | Environment issues |
| Feb 14-17 | V50-V80 | 34-39 | Many experiments | Inconsistent |
| Feb 19 | V86 | 38/50 | V40 params restored | Partial recovery |
| Mar 2 | V111 | 40/50 | 5-component weighted entropy | Stable baseline |
| Mar 10 | V116 | 38/50 | Exact V40 clone attempt | Environment changed |
| **Mar 19** | **V125** | **41/50** | temp=1.0 + top_p=0.8 | **Current best** |
| Mar 21 | V126 | 33/50 | top_p=0.9 | REGRESSION (-8) |
| Mar 22 | V127 | 33/50 | Enhanced prompts | REGRESSION (-8) |
| Mar 24 | V128 | 35/50 | Fine-tuned model (huikang) | REGRESSION (-6) |
| Mar 24 | V130 | 36/50 | Mixed temperature | REGRESSION (-5) |
| Mar 25 | V131 | 35/50 | Combined 6 techniques | REGRESSION (-6) |
| Mar 26 | V132 | 41/50 | Remove top_p | No change |
| Mar 27 | V133 | 41/50 | Answer verification | No change |
| Mar 28 | V134 | 37/50 | Self-refinement | REGRESSION (-4) |
| Mar 29 | V135 | 35/50 | 131K context | REGRESSION (-6) |
| Mar 29 | V136 | 35/50 | Simple mean entropy | REGRESSION (-6) |
| Mar 30 | V137 | TBD | 12 attempts | Testing |
| Mar 30 | V138 | TBD | Untested approaches | Testing |

### Score Timeline Visualization

```
Score over time:

42 |    *                                           <- V40 (Feb 6)
41 |                              * * *             <- V125, V132, V133
40 |                        *                       <- V111
39 |  * *   *                                       <- Early versions
38 |      *     *       *                           <- V86, V116
37 |                                    *           <- V134
36 |                            *                   <- V130
35 |                          * *         * *       <- V128, V131, V135, V136
34 |   *                                            <- V41
33 |                      * *                       <- V126, V127
   +--------------------------------------------------
     Feb 5   Feb 15   Feb 25   Mar 5   Mar 15   Mar 25
```

---

## 3. What Works (Proven Techniques) <a name="3-what-works"></a>

### 3.1 Temperature = 1.0 (High Diversity)

**Evidence:**
- V40 (42/50): temp=1.0
- V125 (41/50): temp=1.0
- V41 (34/50): temp=0.5 → REGRESSION

**Why it works:**
Higher temperature creates more diverse solutions across 8 parallel attempts. The voting mechanism then selects the best answer from this diverse pool. Lower temperature causes all attempts to converge to similar (potentially wrong) answers.

```python
# GOOD: High diversity for voting
temperature = 1.0

# BAD: Too focused, loses diversity
temperature = 0.5  # V41 scored 34/50
```

### 3.2 Five-Component Weighted Entropy

**Evidence:**
- V111 (40/50): 5-component entropy
- V125 (41/50): 5-component entropy
- V136 (35/50): Simple mean entropy → REGRESSION (-6)

**The 5 components:**

```python
def compute_weighted_entropy(logprobs):
    # 1. Base mean entropy (30%)
    mean_ent = sum(entropies) / n
    
    # 2. Variance penalty (20%) - penalize inconsistent confidence
    std_dev = sqrt(variance)
    
    # 3. Position-weighted (40%) - recent tokens matter more
    decay_factor = 0.995
    position_weighted = exponential_decay_weighted_mean(entropies)
    
    # 4. High entropy penalty (30% * 3.0) - penalize uncertain stretches
    high_ent_ratio = count(e > 2.0) / n
    
    # 5. Low entropy streak bonus (-10%) - reward consistent confidence
    streak_bonus = -0.1 * (max_low_ent_streak / n)
    
    return (0.3 * mean_ent + 
            0.4 * position_weighted +  # Most important!
            0.2 * std_dev + 
            0.3 * high_ent_ratio * 3.0 + 
            streak_bonus)
```

**Why it works:**
Position-weighted entropy (40% weight) emphasizes confidence in the final answer tokens. Mathematical solutions often have uncertain exploration early but confident conclusions.

### 3.3 Simple Prompts (V40 Style)

**Evidence:**
- V40 (42/50): 3-line simple prompts
- V125 (41/50): Complex multi-paragraph prompts
- V127 (33/50): Enhanced verbose prompts → REGRESSION (-8)

**V40's winning prompts:**

```python
# SIMPLE (V40 - 42/50)
system_prompt = (
    'You are a world-class IMO competitor. '
    'The final answer must be 0-99999. '
    'Place answer inside \\boxed{}.'
)

tool_prompt = (
    'Use this tool to execute Python code. '
    'Use print() to output results.'
)

preference_prompt = 'You have access to math, numpy, sympy.'
```

**V127's failing prompts:**

```python
# VERBOSE (V127 - 33/50) - DON'T DO THIS
system_prompt = '''
You are an elite mathematical problem solver with expertise at the 
International Mathematical Olympiad (IMO) level...

# Problem-Solving Approach:
1. UNDERSTAND: Carefully read and rephrase...
2. EXPLORE: Consider multiple solution strategies...
3. PLAN: Select the most promising approach...
4. EXECUTE: Work through your solution...
5. VERIFY: Check your answer...

# Mathematical Reasoning Principles:
- Break complex problems into smaller...
... (500+ more tokens)
'''
```

**Why simple works:**
- More tokens available for actual reasoning
- Model already knows how to solve math problems
- Verbose instructions can confuse or distract

### 3.4 Optimal Resource Configuration

**Proven settings for H100:**

```python
# Memory
context_tokens = 65536      # NOT 131K!
gpu_memory_utilization = 0.96
kv_cache_dtype = 'fp8_e4m3'
batch_size = 256

# Voting
attempts = 8                # 10+ causes OOM
early_stop = 4              # Stop when 4 agree

# Timeouts
base_problem_timeout = 270  # seconds
high_problem_timeout = 900  # for hard problems
notebook_limit = 17400      # 4.8 hours total
```

**Why 65K not 131K:**
- 131K context reduces concurrency from ~7x to 3.35x
- Fewer parallel attempts complete in time
- V135 with 131K scored 35/50 (-6 regression)

### 3.5 Inverse Entropy Weighted Voting

**The winning formula:**

```python
def select_answer(results):
    ans_weights = defaultdict(float)
    
    for r in results:
        if r['Answer'] is not None:
            # Lower entropy = higher weight
            ans_weights[r['Answer']] += 1.0 / max(r['Entropy'], 1e-9)
    
    # Pick answer with highest total weight
    return max(ans_weights, key=ans_weights.get)
```

**Why it works:**
- Low entropy responses are more confident
- Confident wrong answers are rare; uncertain correct answers are common
- Weighting by 1/entropy amplifies confident votes

---

## 4. What Doesn't Work (Failed Experiments) <a name="4-what-doesnt-work"></a>

### 4.1 Increased Context Length (131K)

| Version | Context | Score | Delta |
|---------|---------|-------|-------|
| V125 | 65536 | 41/50 | baseline |
| V135 | 131072 | 35/50 | **-6** |

**Why it fails:**
```
vLLM server log with 65K context:
"Maximum concurrency for 65,536 tokens per request: 7.11x"

vLLM server log with 131K context:
"Maximum concurrency for 131,072 tokens per request: 3.35x"
```

Half the concurrency means fewer attempts complete, worse voting.

---

### 4.2 Simple Mean Entropy

| Version | Entropy Type | Score | Delta |
|---------|--------------|-------|-------|
| V125 | 5-component weighted | 41/50 | baseline |
| V136 | Simple mean | 35/50 | **-6** |

**Why it fails:**
Simple mean treats all tokens equally. But in math:
- Early tokens: Exploration, naturally uncertain
- Final tokens: Conclusion, should be confident

The 5-component entropy captures this with position weighting.

---

### 4.3 Fine-Tuned Model (huikang)

| Version | Model | Score | Delta |
|---------|-------|-------|-------|
| V125 | gpt-oss-120b | 41/50 | baseline |
| V128 | huikang/gpt-oss-120b-aimo3 | 35/50 | **-6** |

**Why it fails:**
The fine-tuned model was trained with huikang's specific pipeline (131K context, KV cache management, specific prompts). Using it with our pipeline breaks the optimization.

**Lesson:** Fine-tuning requires matching the entire inference pipeline, not just swapping models.

---

### 4.4 Combined Techniques

| Version | Changes | Score | Delta |
|---------|---------|-------|-------|
| V125 | baseline | 41/50 | - |
| V131 | 6 techniques combined | 35/50 | **-6** |

**V131's 6 techniques:**
1. Enhanced prompts
2. Answer verification
3. Temperature annealing
4. Increased attempts
5. Different seeds
6. Extended timeouts

**Why it fails:**
Each technique interacts with others in unpredictable ways. Testing one variable at a time is essential.

---

### 4.5 Self-Refinement

| Version | Technique | Score | Delta |
|---------|-----------|-------|-------|
| V125 | No refinement | 41/50 | baseline |
| V134 | Self-refinement | 37/50 | **-4** |

**The approach:**
```python
# V134: Ask model to check and correct its answer
refinement_prompt = f'''
A student solved this problem and got {answer}.
Please check if this answer is correct.
- If CORRECT, confirm with \\boxed{{{answer}}}
- If WRONG, solve correctly and give right answer
'''
```

**Why it fails:**
The model "corrects" correct answers to wrong ones. It's overconfident in finding errors that don't exist.

---

### 4.6 Answer Verification (Yes/No)

| Version | Technique | Score | Delta |
|---------|-----------|-------|-------|
| V132 | No verification | 41/50 | baseline |
| V133 | CORRECT/WRONG check | 41/50 | **0** |

**Why it doesn't help:**
The model is equally likely to verify wrong answers as correct. Verification adds latency without improving accuracy.

---

### 4.7 High top_p Values

| Version | top_p | Score | Delta |
|---------|-------|-------|-------|
| V125 | 0.8 | 41/50 | baseline |
| V126 | 0.9 | 33/50 | **-8** |
| V132 | None | 41/50 | 0 |

**Why 0.9 fails:**
top_p=0.9 allows too much probability mass, introducing noise. Either 0.8 or no top_p works; 0.9 is a bad middle ground.

---

### 4.8 Enhanced/Verbose Prompts

| Version | Prompt Style | Score | Delta |
|---------|--------------|-------|-------|
| V40 | Simple 3-line | 42/50 | best |
| V125 | Moderate | 41/50 | -1 |
| V127 | Very verbose | 33/50 | **-9** |

**Why verbose fails:**
- Wastes context tokens on instructions
- Model already knows how to solve math
- Instructions can conflict with model's training

---

## 5. Technical Deep Dive <a name="5-technical-deep-dive"></a>

### 5.1 Architecture Overview

```
┌─────────────────────────────────────────────────────────────┐
│                    AIMO3 Solver Pipeline                    │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  ┌─────────────┐    ┌─────────────┐    ┌─────────────┐     │
│  │   Problem   │───>│   vLLM      │───>│  8 Parallel │     │
│  │   Input     │    │   Server    │    │  Attempts   │     │
│  └─────────────┘    └─────────────┘    └──────┬──────┘     │
│                                               │             │
│                     ┌─────────────────────────┘             │
│                     ▼                                       │
│  ┌─────────────────────────────────────────────────────┐   │
│  │              For each attempt:                      │   │
│  │  ┌─────────┐   ┌─────────┐   ┌─────────┐            │   │
│  │  │ Generate│──>│ Python  │──>│ Continue│──> Answer  │   │
│  │  │  Text   │   │ Execute │   │ or Stop │            │   │
│  │  └─────────┘   └─────────┘   └─────────┘            │   │
│  └─────────────────────────────────────────────────────┘   │
│                     │                                       │
│                     ▼                                       │
│  ┌─────────────────────────────────────────────────────┐   │
│  │           Entropy-Weighted Voting                   │   │
│  │  Answer A: 3 votes, entropy 0.5 → weight 6.0        │   │
│  │  Answer B: 2 votes, entropy 0.8 → weight 2.5        │   │
│  │  Winner: A (highest weight)                         │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

### 5.2 vLLM Server Configuration

```python
# Server startup command
cmd = [
    'python', '-m', 'vllm.entrypoints.openai.api_server',
    '--model', '/kaggle/input/gpt-oss-120b/transformers/default/1',
    '--served-model-name', 'gpt-oss',
    '--tensor-parallel-size', '1',
    '--max-num-seqs', '256',              # Batch size
    '--gpu-memory-utilization', '0.96',   # Leave 4% for overhead
    '--dtype', 'auto',
    '--kv-cache-dtype', 'fp8_e4m3',       # Quantized KV cache
    '--max-model-len', '65536',           # Context length
    '--enable-prefix-caching',            # Share prompt prefix
    '--async-scheduling',
    '--disable-log-stats'
]
```

### 5.3 Jupyter Sandbox Pool

```python
# Pre-initialize 16 Jupyter kernels
self.sandbox_pool = queue.Queue()
for _ in range(16):
    sandbox = AIMO3Sandbox(timeout=6)  # 6 second timeout
    sandbox.execute('import math, numpy, sympy, mpmath')
    self.sandbox_pool.put(sandbox)
```

### 5.4 Parallel Attempt Execution

```python
# Launch 8 parallel attempts
with ThreadPoolExecutor(max_workers=16) as executor:
    futures = [
        executor.submit(self._process_attempt, problem, i, deadline)
        for i in range(8)
    ]
    
    # Early stopping when 4 agree
    for future in as_completed(futures):
        result = future.result()
        if result['Answer']:
            votes[result['Answer']] += 1
            if max(votes.values()) >= 4:  # early_stop
                stop_event.set()
                break
```

---

## 6. Gap Analysis: 42 to 47 <a name="6-gap-analysis"></a>

### Current Leaderboard

| Rank | Team | Score | Gap from Us |
|------|------|-------|-------------|
| 1 | ippeiogawa | 46/50 | +4 |
| 2 | nihilisticneuralnet | 44/50 | +2 |
| ... | ... | ... | ... |
| ? | **Us (V40)** | **42/50** | baseline |

### What Top Solutions Do Differently

| Technique | 44/50 Solution | Our V40 (42/50) |
|-----------|----------------|------------------|
| Temperature | **0.5** | 1.0 |
| Entropy | **Simple mean** | 5-component weighted |
| top_p | None | None |
| Prompts | Structured | Simple |

**Key insight:** The 44/50 solution uses temp=0.5 + simple entropy, while we use temp=1.0 + weighted entropy. These are opposite approaches!

### Untested Approaches

| Approach | Risk | Potential |
|----------|------|----------|
| temp=0.5 + weighted entropy | Low | +1-2 points |
| Problem classification | Medium | +1-2 points |
| Larger answer tiebreaker | Very low | +0-1 points |
| Follow-up prompts | Low | +0-1 points |
| MCTS search | High | +2-4 points |
| Different random seeds | Very low | Variance only |

---

## 7. Key Learnings <a name="7-key-learnings"></a>

### The Golden Rules

1. **Test ONE variable at a time**
   - V131 combined 6 changes = -6 regression
   - Isolate variables to understand impact

2. **Simple > Complex for prompts**
   - V40 (simple): 42/50
   - V127 (verbose): 33/50
   - The model knows math; don't over-instruct

3. **Weighted entropy matters**
   - Position weighting captures confidence in conclusions
   - Simple mean loses critical signal

4. **Memory constraints are real**
   - 131K context halves concurrency
   - 10+ attempts causes OOM
   - Respect H100 limits

5. **Fine-tuning requires full pipeline**
   - Model alone scored -6 (V128)
   - Need matching inference setup

6. **Score variance is high**
   - Same config can give 38-42 on different runs
   - Don't over-optimize for single score

7. **"Improvements" often regress**
   - Answer verification: 0 improvement
   - Self-refinement: -4 regression
   - Be skeptical of clever ideas

---

## 8. Recommendations <a name="8-recommendations"></a>

### For Reproducing Our Best Score (42/50)

```python
# V40 configuration
class CFG:
    temperature = 1.0
    min_p = 0.02
    # NO top_p
    
    context_tokens = 65536
    attempts = 8
    early_stop = 4
    
    # Simple prompts
    system_prompt = (
        'You are a world-class IMO competitor. '
        'Final answer must be 0-99999 in \\boxed{}.'
    )
```

### For Pushing Beyond 42/50

1. **Try temp=0.5** (like 44/50 solution) but keep weighted entropy
2. **Implement MCTS** for tree-search over solution paths
3. **Problem classification** with type-specific strategies
4. **Full huikang pipeline** (not just the model)

### What NOT to Do

- Don't use 131K context
- Don't use simple mean entropy
- Don't combine multiple changes
- Don't use verbose prompts
- Don't add verification/refinement steps

---

## Appendix: All Version Configurations

### V40 (42/50) - Best Ever
```python
temperature = 1.0
min_p = 0.02
context_tokens = 65536
attempts = 8
early_stop = 4
base_problem_timeout = 270
entropy = '5-component weighted'
prompts = 'simple 3-line'
```

### V125 (41/50) - Current Best
```python
temperature = 1.0
top_p = 0.8
min_p = 0.02
context_tokens = 65536
attempts = 8
early_stop = 4
base_problem_timeout = 300
entropy = '5-component weighted'
prompts = 'moderate complexity'
```

### V138 (Testing) - Untested Approaches
```python
temperature = 1.0
min_p = 0.02
# NO top_p
context_tokens = 65536
attempts = 8
early_stop = 4
max_follow_ups = 2
tiebreaker = 'larger answer when close'
entropy = '5-component weighted'
prompts = 'simple + code-first hint'
```

---

## Acknowledgments

- **Competition:** AIMO Prize Foundation
- **Model:** Daniel Hanchen (danielhanchen/gpt-oss-120b)
- **References:** 
  - nihilisticneuralnet (44/50 public notebook)
  - huikang (streaming-inference-tool-calling)
  - datasciencegrad (42/50 stable solution)

---

*Last updated: March 30, 2026*